# Experiment 3: Reasoning and Self-Evaluation


In [ ]:
import os
import pandas as pd
import json
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import scipy.stats as st
import matplotlib.cm as cm
import matplotlib.colorbar as colorbar
from statsmodels.stats.contingency_tables import cochrans_q, mcnemar, Table
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
import plotly.graph_objects as go


import itertools

In [ ]:
# ESAP questions with images
q_w_images = [3, 4, 6, 32, 35, 54, 68, 88]

cases = np.delete(np.arange(100), [0] + q_w_images) # valid questions
n_cases = cases.size

# Available options
options = ['A', 'B', 'C', 'D', 'E']
# even for the experiments without letter, manual annotator saved the option using letter.

path_to_results = "./../results"

model_file_name_dict = {'huatuo-o1': 'HuatuoGPT-o1',
                        'diabetica-o1': 'Diabetica-o1',
                        'diabetica-7B': 'Diabetica-7B',
                        'meditron3-8B': 'Meditron3-8B'}

In [ ]:
def evaluate_response_distrib(filepath):
    """ 
    Given the excel file, this function parses and standardize the file entries.
    Returns a counter in the form of a dictionary.
    """
    df_ = pd.read_excel(filepath, header=None, index_col=0)
    
    # everything capital
    df_ = df_.map(lambda x: x.upper() if isinstance(x, str) else x) 
    
    # remove NaN values - this remove extra-columns
    df_ = df_.dropna(axis=1, how='all') 
    if not np.array_equal(df_.index, cases): # we keep all 91 cases
        raise ValueError("We are removing a valid case by mistake! Check Excel file")

    # unique annotations
    # print(df_.values)
    print(np.unique(df_.values))
    unique_elements = np.unique(df_.values) 

    # multiple options selected
    # are indicated as option1-option2: there is a dash
    bm_multiple = ["-" in v_ for v_ in unique_elements] 
    
    bm_hall_none = [True if (v_ == "HALL" or v_ == 'NONE') # hallucination or no response
                    else False for v_ in unique_elements]
    
    bm_others = ~np.logical_or(bm_multiple, bm_hall_none) # correct answers / check (temporary)
    
    idx_multiple = np.where(bm_multiple)[0] # indexes
    idx_hall_none = np.where(bm_hall_none)[0]
        
    dct_counter = {}
    all_ = 0 # to make sure we did not miss any response (91 or multiple for 10 rep)

    # we are creating a new label for multi and hall/none
    if len(idx_multiple) > 0:
        count = 0
        for v_ in unique_elements[idx_multiple]:
            count += (df_.values == v_).sum()
        all_ = count
        dct_counter["MULTI"] = count
            
    if len(idx_hall_none) > 0:
        count = 0
        for v_ in unique_elements[idx_hall_none]:
            count += (df_.values == v_).sum()
        all_ += count
        dct_counter["HALL / NONE"] = count
       
    # we are using the old label for everything else
    for element in unique_elements[bm_others]:
        count = (df_.values == element).sum()
        dct_counter[element] = count
        all_ += count
            
    list_add = ["CHECK", "MULTI", "HALL / NONE", "A", "B", "C", "D", "E"]
    for el_ in list_add:
        if el_ not in dct_counter.keys():
            dct_counter[el_] = 0
    dct_counter["TOTAL"] = all_
        
    return dct_counter

In [ ]:
model_paths_token = {'meditron3-8B-prompt1': 'meditron3-8B_promptID_001_output_4.xlsx',
                     'huatuo-o1-prompt1': 'huatuo-o1_promptID_001_output_9.xlsx',
                     'huatuo-o1-prompt2': 'huatuo-o1_promptID_002_output_11.xlsx',
                     'diabetica-o1-prompt1': 'diabetica-o1_promptID_001_output_2.xlsx',
                     'diabetica-o1-prompt2': 'diabetica-o1_promptID_002_output_3.xlsx',
                     'diabetica-7B-prompt1': 'diabetica-7B_promptID_001_output_3.xlsx',
                     'diabetica-7B-prompt2': 'diabetica-7B_promptID_002_output_4.xlsx',
                     'medfound7B-prompt1': 'medfound7B_promptID_001_output_2.xlsx',
                     'medfound7B-prompt2': 'medfound7B_promptID_002_output_3.xlsx',
                     'clinical-chatgpt-prompt1': 'clinical-chatgpt_promptID_001_output_2.xlsx',
                     'clinical-chatgpt-prompt2': 'clinical-chatgpt_promptID_002_output_3.xlsx',
                     'bloom-7b-prompt1': 'bloom-7B_promptID_001_output_0.xlsx',
                     'bloom-7b-prompt2': 'bloom-7B_promptID_002_output_1.xlsx',
                     'llama31-prompt1': 'llama31_promptID_001_output_6.xlsx',
                     'llama31-prompt2': 'llama31_promptID_002_output_7.xlsx',
                     'qwen2-7b-prompt1': 'qwen2-7b_promptID_001_output_0.xlsx',
                     'qwen2-7b-prompt2': 'qwen2-7b_promptID_002_output_1.xlsx',
                     'huatuo-o1-lms-prompt1': 'huatuo-o1-lms-prompt001.xlsx',
                     'gpt-oss-20b-prompt1': 'gpt-oss-20b-lms-prompt001.xlsx'
                    }

model_paths_no_token = {'huatuo-o1': 'huatuo-o1_promptID_001_output_47.xlsx',
                        'diabetica-o1': 'diabetica-o1_promptID_001_output_40.xlsx',
                        'diabetica-7B': 'diabetica-7B_promptID_001_output_47.xlsx',
                        'meditron3-8B': 'meditron3-8B_promptID_001_output_39.xlsx',
                        'clinical-chatgpt': 'clinical-chatgpt_promptID_001_output_4.xlsx',
                        'medfound7B': 'medfound7B_promptID_001_output_4.xlsx',
                        'bloom-7b': 'bloom-7b_promptID_001_output_2.xlsx',
                        'llama31': 'llama31_promptID_001_output_8.xlsx',
                        'qwen2-7b': 'qwen2-7b_promptID_001_output_2.xlsx'
                       }

In [ ]:
def counter_extensive_dfs(result_folder, dct_models, letter=True):
    """ 
    Used in experiment 1 to a) keep track of distribution of responses
    b) build dataframe with extensive results.
    
    Variables: 
    result_folder: str, folder containing model's output
    dct_models: str, excel file 
    letter: bool, if True, ESAP question, otherwise, letter removed from option
    
    Returns:
    df_counter: pd.DataFrame, distribution of models responses
    df_all: pd.DataFrame extensive (91 outputs)
    """
    list_df = []
    index = []
    counting_outputs = []

    for model_, file_ in dct_models.items():
        if letter:
            filepath = join(path_to_results, model_.split("-prompt")[0], file_)
        else: 
            filepath = join(path_to_results, "NO_LETTERS", file_)
            
        df_ = pd.read_excel(filepath, header=None, index_col=0)
        df_ = df_[1] # we keep first column, others are empty
        list_df.append(df_)
        
        dct_counter = evaluate_response_distrib(filepath)
        counting_outputs.append([dct_counter[k] for k in sorted(dct_counter.keys())])

    # this is for the counter
    columns_name = sorted(dct_counter.keys())
    
    list_model_name = [m_ if letter else f"{m_}_no_letter" for m_ in dct_models.keys()]

    df_counter = pd.DataFrame(data=counting_outputs, 
                              index=list_model_name,
                              columns=columns_name)

    # this is for the Df with 91 responses
    df_all = pd.concat(list_df, axis=1)
    df_all.columns = list_model_name
    
    return df_counter, df_all

In [ ]:
df_counter_prompt_A_B, df_all = counter_extensive_dfs(path_to_results, model_paths_token)

In [ ]:
df_counter_no_let, df_all_no_let = counter_extensive_dfs(path_to_results, model_paths_no_token, letter=False)

In [ ]:
# concatenate all results from experiment 1
df_all = pd.concat([df_all, df_all_no_let], axis=1)

In [ ]:
with open("./../inference/Ped-ESAP.json", "r") as f:
    content = json.load(f)
    
responses = []
labs = []
table = []
for k, v in content.items():
    if len(v['answer']) > 0:
        responses.append(v['answer'][0])
        labs.append(v["labs"]=='Yes')
        table.append(v["table"]=='Yes')
    else:
        responses.append(None)
        labs.append(None)
        table.append(None)
        
ground_truth = pd.DataFrame(data=np.array(responses[:-1]).reshape(-1,1), index=np.arange(1,100),
                           columns=["truth"])
ground_truth = ground_truth.drop(q_w_images)

meta_labs_table = np.hstack((np.array(labs).reshape(-1,1), np.array(table).reshape(-1,1)))
meta_labs_tab_df = pd.DataFrame(data=meta_labs_table[:-1, :], index=np.arange(1,100),
                    columns=["labs", "table"])
meta_labs_tab_df = meta_labs_tab_df.drop(q_w_images)

In [ ]:
# run only once, otherwise we get multiple "truth" columns
df_all = pd.concat([df_all, ground_truth], axis=1)

In [ ]:
# here we keep the value in entry if is one among A, B, C, D, or E
# we report as NaN otherwise
for col in df_all.columns[:-1]:
    df_all[col] = df_all[col].where(df_all[col].isin(options), np.nan)

In [ ]:
bm_valid = ~df_all.isna()

In [ ]:
accuracy_df = df_all[df_all.columns[:-1]].eq(df_all['truth'], axis=0)

accuracy_df.head()

In [ ]:
accuracy_scores = accuracy_df.sum()

In [ ]:
models_col = df_all.columns[:-1] # we exclude the truth column

In [ ]:
results = pd.concat([accuracy_scores[models_col], bm_valid[models_col].sum()], axis=1)
results.columns = ['# correct', '# interpretable']

In [ ]:
os.makedirs("tables", exist_ok=True)
os.makedirs("plots", exist_ok=True)


# Experiment 3: Check Reasoning and Self-Evaluation

In [ ]:
os.makedirs("plots", exist_ok=True)

In [ ]:
physician_eval = {"huatuo-o1_promptID_001_output_9": [3, 4, 4, 4, 3, 2, 2, 3], 
                  "huatuo-o1_promptID_002_output_11": [2, 2, 2, 2],
                  "diabetica-o1_promptID_001_output_2": [],
                  "diabetica-o1_promptID_002_output_3": []}

dict_fileid = {"huatuo-o1": {1: 9,
                             2: 11},
              "diabetica-o1": {1: 2,
                               2: 3}
              }

In [ ]:
contingency_table_dict = {}
# we save a contigency table with frequency of selection of ESAP vs model response

filtered_df_dict = {}
# we save dataframes whenever the model prefers its previous response over ESAP

missing_choice = {}

for model_name in dict_fileid:
    
    path_to_reasoning = join(path_to_results, "REASONING", model_name)
    contingency_table_dict[model_name] = {}
    filtered_df_dict[model_name] = {}
    missing_choice[model_name] = {}
    
    
    for prompt, file_id in dict_fileid[model_name].items():
        
        experiment_name = f"{model_name}_promptID_00{prompt}_output_{file_id}"

        list_df_esap = []
        for esap in [1, 2]: # ground truth in first or second position

            model_exp = {1: 2, 2: 1}[esap]  # model reasoning in other location
            model_and_exp_name = experiment_name.split("_output")[0]

            filename = f"reasoning_ESAP{esap}_{model_name}_{experiment_name}.xlsx"
            df_reason = pd.read_excel(join(path_to_reasoning, filename),
                          header=None, index_col=0)
            n_cols = df_reason.shape[1] 
            if n_cols >= 3: # eliminate nan columns
                df_reason = df_reason.drop([i for i in range(3, n_cols+1)], axis=1)

            df_reason.columns = [f"ESAP {esap} preferred", f"Agreement {esap}"]
            df_reason[f"ESAP {esap} preferred"] = df_reason[f"ESAP {esap} preferred"].map({esap: True, model_exp: False})
            list_df_esap.append(df_reason)
            
        df_ = pd.concat([pd.concat(list_df_esap, axis=1), accuracy_df[model_name+f"-prompt{prompt}"]], axis=1)

        # df_[df_.columns[-1]] model response from experiment 1 was correct
        df_subset = df_[~df_[df_.columns[-1]]] # model from experiment 1 was wrong

        # Drop rows where 'ESAP preferred' is NaN
        filtered_df = df_subset[(df_subset['ESAP 1 preferred'].notna()) 
                                & (df_subset['ESAP 2 preferred'].notna())]

        # Count dropped rows
        num_dropped = len(df_subset) - len(filtered_df)

        # Create contingency table
        contingency_table = pd.crosstab(filtered_df['ESAP 1 preferred'], filtered_df['ESAP 2 preferred'])
        contingency_table_dict[model_name][prompt] = contingency_table
        
        filtered_df_dict[model_name][prompt] = filtered_df
        
        # print(f"Rows with missing response in at least one of the two runs': {num_dropped}")        
        missing_choice[model_name][prompt] = num_dropped

In [ ]:
def plot_boxplot_agreement_levels(dict_wrong_answers, model_name, prompt):
    filtered_df = dict_wrong_answers[model_name][prompt]
    prompt_label = ["A", "B"]

    cond1 = filtered_df['ESAP 1 preferred'].astype(bool)
    cond2 = filtered_df['ESAP 2 preferred'].astype(bool)

    nn = filtered_df[(~cond1 & ~cond2)][['Agreement 1', 'Agreement 2']].mean(axis=1)
    yy = filtered_df[(cond1 & cond2)][['Agreement 1', 'Agreement 2']].mean(axis=1)
    n1y2 = filtered_df[(~cond1 & cond2)][['Agreement 1', 'Agreement 2']].mean(axis=1)
    y1n2 = filtered_df[(cond1 & ~cond2)][['Agreement 1', 'Agreement 2']].mean(axis=1)

    boxprops = dict(linewidth=2)
    medianprops = dict(linewidth=2)
    whiskerprops = dict(linewidth=2)
    capprops = dict(linewidth=2)
    flierprops = dict(marker='o', markersize=5, linestyle='none')

    fig, ax = plt.subplots(ncols=4, sharey=True)
    ax[0].set_ylim([0,5.2])
    for a in ax:
        a.spines['bottom'].set_linewidth(3)
        a.spines['left'].set_linewidth(3)
        a.spines['top'].set_linewidth(3)
        a.spines['right'].set_linewidth(3)
        a.tick_params(width=3)

    nn.plot.box(ax=ax[3], label='ESAP\nnever',
                boxprops=boxprops, medianprops=medianprops,
                whiskerprops=whiskerprops, capprops=capprops, flierprops=flierprops,
                title=f"$n={nn.size}$", fontsize=16)

    yy.plot.box(ax=ax[0], label='ESAP\nboth',
                boxprops=boxprops, medianprops=medianprops,
                whiskerprops=whiskerprops, capprops=capprops, flierprops=flierprops,
                title=f"$n={yy.size}$", fontsize=14)
    n1y2.plot.box(ax=ax[1], label='ESAP\nsecond',
                  boxprops=boxprops, medianprops=medianprops,
                  whiskerprops=whiskerprops, capprops=capprops, flierprops=flierprops,
                  title=f"$n={n1y2.size}$", fontsize=14)
    y1n2.plot.box(ax=ax[2], label='ESAP\nfirst', 
                  boxprops=boxprops, medianprops=medianprops,
                  whiskerprops=whiskerprops, capprops=capprops, flierprops=flierprops,
                  title=f"$n={y1n2.size}$", fontsize=14)
    ax[0].set_ylabel("Agreement Level", fontsize=18)

    for i_ in range(4):
        ax[i_].set_title(ax[i_].get_title(), fontsize=16)   # increase title size


    fig.text(0.5, 0.99, f"Comparing ESAP with \nWrong Outcomes (Prompt {prompt_label[prompt-1]})", 
             ha='center', fontsize=18);

    plt.subplots_adjust(bottom=0.15, top=0.9)

    plt.tight_layout()

    fig.text(0.5, -0.05, "Picking ESAP Explanation as Best", ha='center', fontsize=20)
    # plt.savefig(f"plots/{model_name}_{prompt}_boxplot.pdf", bbox_inches='tight')
    plt.show()

In [ ]:
for model_name in dict_fileid:
    for prompt in dict_fileid[model_name]:
        print(f"{model_name} for prompt {prompt}")
        plot_boxplot_agreement_levels(filtered_df_dict, model_name, prompt)

In [ ]:
contingency_table_dict['huatuo-o1'][1]

In [ ]:
# Labels for each group (model + prompt)
labels = ['A', 'B'] 

outcome = ["Hall / None",
           "ESAP \nnever",
           "ESAP \nsecond",
           "ESAP \nfirst",
           "ESAP \nboth"
]

data_dict = {}
percentage_dict = {}

for model_name in dict_fileid:
    data_dict[model_name] = []
    percentage_dict[model_name] = []
    
    for prompt in dict_fileid[model_name]:
        tmp_data = np.append(missing_choice[model_name][prompt], 
                             np.ravel(contingency_table_dict[model_name][prompt].values))
        data_dict[model_name].append(tmp_data)
        percentage_dict[model_name].append(tmp_data / tmp_data.sum())

    data_dict[model_name] = np.array(data_dict[model_name])
    percentage_dict[model_name] = 100 * np.array(percentage_dict[model_name])

# Define outcome colors (5 outcomes)
colors = ['lightgray', 'orangered', 'gold', 'khaki', 'teal'] #  

# Create plot
nrows = len(dict_fileid.keys())
fig, ax = plt.subplots(figsize=(10, 6), ncols=1, nrows=nrows, sharex=True)

for j_, model_name in enumerate(dict_fileid.keys()):

    ax[j_].spines['bottom'].set_linewidth(3)
    ax[j_].spines['left'].set_linewidth(3)
    ax[j_].spines['top'].set_linewidth(3)
    ax[j_].spines['right'].set_linewidth(3)
    ax[j_].tick_params(width=3)
    
    for k_, (percentage_, data_) in enumerate(zip(percentage_dict[model_name], data_dict[model_name])):

        left = 0
        for i in range(data_.size):  # for each outcome
            value = data_[i]
            perc = percentage_[i]
            if perc > 2:  #  skip annotation on short segments
                ax[j_].text(left + value / 2, k_, f"{perc:.0f}%", va='center', 
                                ha='center', fontsize=18, color='black')
            
            label = outcome[i] if (k_ == 0 and j_ == 0) else None
            
            ax[j_].barh(labels[k_], data_[i], left=left, color=colors[i], alpha=.7, 
                        label=label)
            left += data_[i]

        tick = ax[j_].yaxis.get_major_ticks()
        for t in tick:
            t.label1.set_fontsize(17) 
        
        ax2 = ax[j_].secondary_yaxis('right')
        
        ax2.set_ylabel(f"{model_file_name_dict[model_name]}", rotation=270, labelpad=20, fontsize=17)
        ax2.set_yticks([])
        ax2.tick_params(length=0)    # hide its ticks if you’d like

    tick = ax[1].xaxis.get_major_ticks()
    for t in tick:
        t.label1.set_fontsize(18) 

    # Add legend and labels
    ax[1].set_xlabel('# Questions - Only Those Answered Incorrectly from Exp 1', fontsize=20)
    # ax[1].set_xlim(0, 61)
    ax[0].legend(bbox_to_anchor=(0.5, 1.4), loc='upper center', ncol=5, fontsize=15)

    plt.tight_layout()
    # plt.savefig("plots/model_re-eval_given_gt.pdf")
plt.show()